# Project: Investigate the movies data

## My questions
1. Do films that cost more money to make also earn more money?
2. Which genre groups get the best ratings, and which earn the most money?
3. Do critics and normal viewers like the same films?

## Part 1 : Read and understand the data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [2]:
movies_raw = pd.read_csv("data/movies/movies.csv")
movies = movies_raw.copy()
genre_groups = pd.read_csv("data/movies/genre_groups.csv")

In [3]:
type(movies)

pandas.DataFrame

In [4]:
print(movies.shape)
print(genre_groups.shape)

(3201, 16)
(12, 2)


### Look at the first rows

In [5]:
movies.head()

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
0,The Land Girls,146083.0,146083.0,NaN,8000000.0,Jun 12 1998,R,NaN,Gramercy,NaN,NaN,NaN,NaN,NaN,6.1,1071.0
1,"First Love, Last Rites",10876.0,10876.0,NaN,300000.0,Aug 07 1998,R,NaN,Strand,NaN,Drama,NaN,NaN,NaN,6.9,207.0
2,I Married a Strange Person,203134.0,203134.0,NaN,250000.0,Aug 28 1998,NaN,NaN,Lionsgate,NaN,Comedy,NaN,NaN,NaN,6.8,865.0
3,Let's Talk About Sex,373615.0,373615.0,NaN,300000.0,Sep 11 1998,NaN,NaN,Fine Line,NaN,Comedy,NaN,NaN,13.0,NaN,NaN
4,Slam,1009819.0,1087521.0,NaN,1000000.0,Oct 09 1998,R,NaN,Trimark,Original Screenplay,Drama,Contemporary Fiction,NaN,62.0,3.4,165.0


### Columns, how many values each column has and types:

In [6]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Title                   3200 non-null   str    
 1   US Gross                3194 non-null   float64
 2   Worldwide Gross         3194 non-null   float64
 3   US DVD Sales            564 non-null    float64
 4   Production Budget       3200 non-null   float64
 5   Release Date            3201 non-null   str    
 6   MPAA Rating             2596 non-null   str    
 7   Running Time min        1209 non-null   float64
 8   Distributor             2969 non-null   str    
 9   Source                  2836 non-null   str    
 10  Major Genre             2926 non-null   str    
 11  Creative Type           2755 non-null   str    
 12  Director                1870 non-null   str    
 13  Rotten Tomatoes Rating  2321 non-null   float64
 14  IMDB Rating             2988 non-null   float64
 15

- The table has 3201 rows and 16 columns. 
- **One row is one film.**
- The columns are about money (US Gross, Worldwide Gross, US DVD Sales, Production Budget), the release (Release Date, Distributor, MPAA Rating), the film itself (Running Time min, Source, Major Genre, Creative Type, Director) and the ratings (Rotten Tomatoes Rating, IMDB Rating, IMDB Votes).
- Data types :
    - **Numbers (float64), 8 columns:** US Gross, Worldwide Gross, US DVD Sales, Production Budget, Running Time min, Rotten Tomatoes Rating, IMDB Rating and IMDB Votes.
    - **Text (str), 8 columns:** Title, Release Date, MPAA Rating, Distributor, Source, Major Genre, Creative Type and Director.
    - **Release Date is text, but it should be a date.** I fix that in Part 2.


**Columns I will use:** Title, Production Budget, US Gross, Worldwide Gross, Release Date, Major Genre, Rotten Tomatoes Rating, IMDB Rating and IMDB Votes.
I will not use US DVD Sales or Running Time min much, because most of their values are missing.

## Part 2 : Clean at least three different problems

### Problem 1 : Release Date column has wrong data type as str, so converting to datetime

In [7]:
movies["Release Date"] = pd.to_datetime(movies["Release Date"], errors="coerce")

In [8]:
print(movies["Release Date"].dtype)

datetime64[us]


In [9]:
movies["Release Date"].isna().sum()

np.int64(0)

- Problem: Release Date was text (str), not a date.
- Rows: 3201 (all films)
- Fix: converted it with pd.to_datetime. 0 dates became NaT, so every date could be read.

### Problem 2 : Some years were 100 years too late

In [10]:
movies["Release Date"].min()

Timestamp('1928-12-31 00:00:00')

In [11]:
movies["Release Date"].max()

Timestamp('2046-12-31 00:00:00')

In [12]:
movies["Release Date"].dt.year.value_counts().sort_index().tail(20)

Release Date
2007    164
2008    160
2009    130
2010     92
2011      2
2015      1
2016      2
2020      1
2025      2
2027      1
2033      3
2036      2
2037      1
2039      2
2040      1
2041      1
2042      1
2043      1
2044      1
2046      2
Name: count, dtype: int64

In [13]:
(movies["Release Date"].dt.year > 2011).sum()

np.int64(22)

- I checked a few on the internet. For example: Gone with the Wind is from 1939, not 2039. 
- The 2 films in 2011 are real.
- One film in 2015 is an old film.


Hence using years after 2011.

In [14]:
movies[movies["Release Date"].dt.year > 2011][["Title", "Release Date"]]

,Title,Release Date
9,Duel in the Sun,2046-12-31
15,Over the Hill to the Poorhouse,2020-09-17
16,Wilson,2044-08-01
26,"20,000 Leagues Under the Sea",2016-12-24
33,42nd Street,2033-03-09
85,Ben-Hur,2025-12-30
90,The Best Years of Our Lives,2046-11-21
102,The Big Parade,2025-01-01
120,The Birth of a Nation,2015-02-08
174,"Charge of the Light Brigade, The",2036-10-20


- So 22 rows have wrong year.
- I checked 3 films and all were exactly 100 years off. I assume the other 19 have the same mistake.

In [15]:
wrong_year = movies["Release Date"].dt.year > 2011

### Fix the 22 years

In [16]:
movies.loc[wrong_year, "Release Date"] = movies.loc[wrong_year, "Release Date"] - pd.DateOffset(years=100)

### verify those 22 years

In [17]:
(movies["Release Date"].dt.year > 2011).sum()

np.int64(0)

In [18]:
movies["Release Date"].max()

Timestamp('2011-12-23 00:00:00')

- Problem: some years were 100 years too late (e.g. Gone with the Wind 2039 instead of 1939).
- Rows: 22
- Fix: subtracted 100 years. Now 0 films are after 2011.

### Problem 3 : Some films have 0 in US Gross and Worldwide Gross

In [19]:
movies[["US Gross", "Worldwide Gross"]].describe()

,US Gross,Worldwide Gross
count,3.194000e+03,3.194000e+03
mean,4.400209e+07,8.534340e+07
std,6.255531e+07,1.499473e+08
min,0.000000e+00,0.000000e+00
25%,5.493221e+06,8.031285e+06
50%,2.201947e+07,3.116893e+07
75%,5.609176e+07,9.728380e+07
max,7.601676e+08,2.767891e+09


In [20]:
(movies[["US Gross", "Worldwide Gross"]] == 0).sum()

US Gross           66
Worldwide Gross    47
dtype: int64

In [21]:
movies["US Gross"] = movies["US Gross"].where(movies["US Gross"] != 0)

In [22]:
movies["Worldwide Gross"] = movies["Worldwide Gross"].where(movies["Worldwide Gross"] != 0)

In [25]:
(movies[["US Gross", "Worldwide Gross"]] == 0).sum()

US Gross           0
Worldwide Gross    0
dtype: int64

- Problem: some films have 0 in US Gross and Worldwide Gross, No film shown in cinemas earns exactly 0, so 0 here is not a real value. It means the number is unknown, so I replaced it with NaN.
- Rows: 
    - US Gross : 66
    - Worldwide Gross : 47
- Fix: Replaced those 0 with NaN

## Part 3 : Handle the missing values

### count missing values in every column

In [26]:
movies.isna().sum()

Title                        1
US Gross                    73
Worldwide Gross             54
US DVD Sales              2637
Production Budget            1
Release Date                 0
MPAA Rating                605
Running Time min          1992
Distributor                232
Source                     365
Major Genre                275
Creative Type              446
Director                  1331
Rotten Tomatoes Rating     880
IMDB Rating                213
IMDB Votes                 213
dtype: int64

Many values are missing in columns: US DVD Sales, Running Time min and Director with 2637, 1992, 1331 of 3201. They also don't help any of my questions, so I drop them.

### Drop the columns with many missing values

In [27]:
movies = movies.drop(columns=["US DVD Sales", "Running Time min", "Director"])

In [28]:
movies.shape

(3201, 13)

### Keep Rotten Tomatoes Rating

I will keep the Rotten Tomatoes Rating column even though it has 880 missing values, because I need it for question 3: Do critics and normal viewers like the same films?

I will leave them as NaN, because filling would make up critic scores, and dropping the rows would lose those films from my other questions. pandas skips NaN automatically.